# 4D-Humans VR: Google Colab 3D Generation Server Runner (TRELLIS.2)

This notebook sets up the environment and launches the Microsoft **TRELLIS.2 Image-to-3D** generation server (4B parameters), tunneling it to your local frontend application using a secure Ngrok bridge.

### Step 1: Install Dependencies
Colab comes with PyTorch pre-installed. Run the cell below to install our server dependencies and the custom CUDA extensions required by the TRELLIS.2 pipeline.

In [ ]:
# Always reset the working directory to /content on cell reruns to prevent double-nesting
%cd /content

# 1. Clean up and clone your repository (Replace with your actual GitHub repo URL if different)
!rm -rf BISAG
!git clone https://github.com/anantj09/BISAG.git
%cd BISAG/Backend

# 2. Install base PyPI dependencies
!pip install -r colab/requirements_colab.txt

# 3. Install spconv and clone/setup Microsoft TRELLIS.2
!pip install spconv

# Clone TRELLIS.2 repository (and build custom extensions)
!rm -rf /content/TRELLIS
!git clone --recursive https://github.com/microsoft/TRELLIS.2.git /content/TRELLIS

# Install C++ and CUDA libraries (Eigen3 headers are required by o-voxel)
!sudo apt-get -y install libeigen3-dev

# Install CuMesh and FlexGEMM with no-build-isolation first (required by o-voxel)
!pip install git+https://github.com/JeffreyXiang/CuMesh.git --no-build-isolation
!pip install git+https://github.com/JeffreyXiang/FlexGEMM.git --no-build-isolation

# If the submodule clone failed or is incomplete, use system Eigen headers via symlink
# Note: We symlink /usr/include/eigen3 (which contains the 'Eigen' folder) so that o-voxel/third_party/eigen/Eigen matches
!if [ ! -d "/content/TRELLIS/o-voxel/third_party/eigen/Eigen" ]; then rm -rf /content/TRELLIS/o-voxel/third_party/eigen && ln -s /usr/include/eigen3 /content/TRELLIS/o-voxel/third_party/eigen; fi

# Install o-voxel with no-build-isolation
!pip install /content/TRELLIS/o-voxel --no-build-isolation

# Install nvdiffrast using no-build-isolation
!pip install git+https://github.com/NVlabs/nvdiffrast.git --no-build-isolation

# Install diff-gaussian-rasterization for 3D Gaussian Splatting capabilities with no-build-isolation
!pip install git+https://github.com/graphdeco-inria/diff-gaussian-rasterization.git --no-build-isolation

### Step 2: Configure Hugging Face Token & Request Model Access
TRELLIS.2 uses Meta's **DINOv3** model (`facebook/dinov3-vitl16-pretrain-lvd1689m`) as a feature extractor, which is a gated repository.

1. Log in to Hugging Face, visit [facebook/dinov3-vitl16-pretrain-lvd1689m](https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m), and click **"Request Access"** (access is granted instantly).
2. Generate a **Read** token in your Hugging Face [Access Tokens settings](https://huggingface.co/settings/tokens).
3. In the left-hand sidebar of Google Colab, click the **Secrets (key icon 🔑)** tab.
4. Add a new secret named `HF_TOKEN`, paste your Hugging Face access token, and toggle **Notebook access** ON.

### Step 3: Open Tunnel and Start Server
1. Go to your [Ngrok Dashboard](https://dashboard.ngrok.com/) and copy your free authtoken.
2. Paste it in place of `YOUR_NGROK_AUTHTOKEN` in the code cell below.
3. Run the cell. Once the setup completes, copy the generated public HTTPS URL (e.g. `https://xxxx.ngrok-free.app`).
4. Open the Settings panel on your local page and paste this URL to activate Colab generation!

In [ ]:
# 4. Launch the modular server and Ngrok tunnel
!python -m colab.main_colab --ngrok-token "YOUR_NGROK_AUTHTOKEN"